[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/building-rag-pipelines/blob/main/notebooks/07_advanced_rag.ipynb)

# Building RAG Pipelines
## Notebook 07: Explore Advanced RAG Concepts
**Duration:** 15 min &nbsp;|&nbsp; **Mode:** Conceptual

> Taught **WHY → WHAT → HOW**. We keep asking *"What happens if this step is poorly
> designed?"* and we **predict before we run** and **compare outputs**. LangChain is
> shown as a **parallel mapping** — it abstracts mechanics but not design decisions.

![pipeline](https://dummyimage.com/1000x70/1f2937/ffffff&text=Loading+%E2%86%92+Chunking+%E2%86%92+Retrieval+%E2%86%92+Augmentation+%E2%86%92+Generation+%E2%86%92+Evaluation)

In [ ]:
# ============================================================
# COLAB BOOTSTRAP — run this cell first. (Same as every notebook.)
# ============================================================
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/baluragala/building-rag-pipelines.git"  # INSTRUCTOR: set this

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

_pip("numpy", "openai", "tiktoken", "rank-bm25", "beautifulsoup4", "pypdf",
     "langchain-community", "langchain-text-splitters", "langchain-openai", "faiss-cpu")
try:
    import rag_pipeline
except ModuleNotFoundError:
    if IN_COLAB:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
        if os.path.isdir("building-rag-pipelines"):
            sys.path.insert(0, "building-rag-pipelines")
        else:
            print("Clone failed. Upload `rag_pipeline/` + `data/` via the Colab file browser, then re-run.")
    else:
        sys.path.insert(0, os.path.abspath(".."))
    import rag_pipeline

def data_path(*parts):
    for base in ("data", "../data", "building-rag-pipelines/data"):
        p = os.path.join(base, *parts)
        if os.path.exists(p):
            return p
    return os.path.join("data", *parts)

print("rag_pipeline", rag_pipeline.__version__, "ready.  Colab:", IN_COLAB)

In [ ]:
# ============================================================
# CHOOSE YOUR PROVIDERS  (OpenAI is the default)
# ============================================================
# Default stack = OpenAI: gpt-4o-mini (LLM) + text-embedding-3-small (embeddings).
# In Colab the key is read automatically from the Colab SECRETS manager:
#   left sidebar -> key icon -> add a secret named OPENAI_API_KEY
#   -> toggle "Notebook access" ON  -> re-run this cell.
# If no key is found anywhere, we fall back to the offline MOCK so the notebook
# still runs end-to-end.
import os

def _load_openai_key():
    if os.getenv("OPENAI_API_KEY"):
        return True
    try:  # Colab Secrets
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
            return True
    except Exception:
        pass  # not in Colab, secret missing, or access not granted
    return False

if _load_openai_key():
    os.environ.setdefault("RAG_LLM_PROVIDER", "openai")
    os.environ.setdefault("RAG_EMBED_PROVIDER", "openai")
    print("OpenAI key found -> using the OpenAI stack.")
else:
    os.environ["RAG_LLM_PROVIDER"] = "mock"
    os.environ["RAG_EMBED_PROVIDER"] = "mock"
    print("No OPENAI_API_KEY found -> using the offline MOCK providers.\n"
          "In Colab: add a Secret named OPENAI_API_KEY (key icon, left sidebar),\n"
          "enable Notebook access, and re-run this cell to switch to OpenAI.")

from rag_pipeline import config
print(config.current_config())

## WHY — basic RAG is a floor, not a ceiling

The modular pipeline you built is **Naive RAG**. It's the right default, but real
systems hit its limits: ambiguous queries, questions that need multiple retrieval
hops, structured data, or knowledge that changes constantly. This notebook is a
map of where to go next — *positioned as next steps*, not today's deep dives.

## WHAT — the three RAG architectures

| Architecture | What changes |
|--------------|--------------|
| **Naive RAG** | retrieve → stuff → generate. One shot, fixed k. (What we built.) |
| **Advanced RAG** | adds pre-retrieval (query rewriting, expansion) and post-retrieval (reranking, compression) steps around the same core. |
| **Modular RAG** | retrieval, reranking, routing, memory, generation as *swappable modules* orchestrated by control logic — our `RAGPipeline` is a small step toward this. |

## WHAT — techniques worth knowing (breadth, not depth)

- **Agentic RAG / tool-augmented retrieval** — an agent decides *when* and *what*
  to retrieve, and can call tools. (We only *name* it here — agents are a later topic.)
- **Multi-hop RAG & query decomposition** — break a complex question into
  sub-questions, retrieve for each, then compose. (Our "SSO + 99.99% SLA" question
  is a baby multi-hop case.)
- **Retrieval over structured data** — query SQL databases or **knowledge graphs**.
  **Graph RAG** retrieves *subgraphs* of entities/relations, so it answers
  relationship questions ("who reports to whom") that flat chunks can't.
- **Adaptive retrieval** — **dynamic k** (retrieve more only when confidence is low)
  and **query rewriting** (clean up a messy user query before retrieval).
- **Emerging trends** — **long-context models vs RAG** (do you still need retrieval
  if the model reads 1M tokens? Usually yes: cost, freshness, provenance, precision),
  and **caching** (embedding caches, retrieval caches, prompt caches) for latency/cost.

## HOW — two tiny illustrations

These are *sketches* to make the ideas concrete, not production techniques.

In [ ]:
from rag_pipeline import config
llm = config.get_llm()

# 1) QUERY REWRITING (a pre-retrieval Advanced-RAG step): turn a messy query into a
#    clean, retrieval-friendly one before it ever hits the retriever.
messy = "growth plan how much $$ per mo??"
rewrite = llm.generate(
    "Rewrite the user's messy question as a single clear, formal search query. "
    "Reply with only the rewritten query.\n\nUser: " + messy)
print("original:", messy)
print("rewritten:", rewrite)

In [ ]:
# 2) QUERY DECOMPOSITION (multi-hop): split a compound question into sub-questions.
compound = "Which plan supports SSO and what uptime does it guarantee?"
subqs = llm.generate(
    "Break the QUESTION into the minimal list of standalone sub-questions, one per "
    "line.\n\nQUESTION: " + compound)
print("Sub-questions to retrieve for:\n", subqs)

## WHAT — when to move beyond basic RAG

Reach for advanced techniques when you observe a *specific* failure the basic
pipeline can't fix by tuning:

| Symptom | Consider |
|---------|----------|
| Complex questions needing several facts | multi-hop / query decomposition |
| Relationship / "who-connects-to-whom" questions | Graph RAG / knowledge graphs |
| Answers depend on live/structured records | SQL retrieval, tool-augmented RAG |
| Vague user queries | query rewriting / expansion |
| Over-retrieving simple questions | adaptive / dynamic k |
| Latency & cost pain at scale | caching layers |

> **The discipline stays the same:** add a technique because a *measured* failure
> demands it — not because it's fashionable. Every addition is another design
> decision that must earn its place on the eval set.

## Recap
- Naive → Advanced → Modular is a progression of added control, not a rewrite.
- Multi-hop, Graph RAG, structured retrieval, adaptive retrieval, caching — each
  targets a specific limitation of the basic pipeline.
- Long context complements, rarely replaces, retrieval.

**Next → Notebook 08 (Conclusion):** wire the full pipeline, watch design choices
change the output, and debug a *bad* RAG output stage by stage.